In [ ]:
# ===================== DAY 1: data, tokenizer, batches =====================
import torch

torch.manual_seed(1337)

# --- hyperparameters (we'll grow this list as we go) ---
batch_size = 4   # how many independent sequences we process in parallel
block_size = 8   # max context length the model ever sees

# --- 1. load the data ---
# first run in a notebook:  !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters:", len(text))  # expect 1115394

# --- 2. vocabulary: every unique character is a token ---
chars = sorted(list(set(text)))
vocab_size = len(chars)  # expect 65
print(''.join(chars), vocab_size)

# --- 3. tokenizer: char <-> int ---
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]             # string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l])    # list of ints -> string
assert decode(encode("hii there")) == "hii there"   # round trip must be lossless

# --- 4. whole dataset as one long tensor of ints ---
data = torch.tensor(encode(text), dtype=torch.long)  # shape (1115394,)

# --- 5. train/val split: first 90% train, last 10% val ---
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# --- 6. batching ---
# A chunk of block_size+1 chars gives block_size training examples:
# x = chars [0..7], y = chars [1..8]; at position t, input is x[:t+1], target is y[t].
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))   # random start positions
    x = torch.stack([d[i:i + block_size] for i in ix])        # (B, T) inputs
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # (B, T) targets, shifted by 1
    return x, y

xb, yb = get_batch('train') # xb training set and yb target. 
print(xb.shape, yb.shape)  # both torch.Size([4, 8])
# ===========================================================================

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

         
        self.embedding  = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # TODO: logits from the table, shape (B, T, C)
        logits = self.embedding(idx)
        


        # TODO: if targets is None -> loss = None
        #       else reshape logits and targets (your answer to question 3), compute cross_entropy
        if(targets == None):
            loss = None   
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)


        return logits, loss


    def generate(self, idx, max_new_tokens):
        # TODO: loop max_new_tokens times:
        #   get logits, take the last position only, softmax, multinomial, append to idx
        for _ in range(max_new_tokens):
            logits, loss = self.forward(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, ix), dim=1)
            

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)                     # parameter 1                                    # parameter 2
print(      decode(     m.generate(idx = torch.zeros((1,1), dtype=torch.long),      max_new_tokens=100)[0].tolist())     )

In [ ]:
# TODO: create the optimizer (AdamW, lr=1e-3) from the model's parameters
optmizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# nn modules knows its parameters by nn.embedding() where nn is imported as torch.nn

batch_size = 32

for step in range(10000):   # start small, then raise it
    # TODO: 1. get a train batch
    xb, yb = get_batch('train')
    # TODO: 2. forward pass -> logits, loss
    logits, loss = m(xb, yb)
    # TODO: 3. clear old gradients
    optmizer.zero_grad(set_to_none=True)
    # TODO: 4. backward pass
    loss.backward()
    # TODO: 5. optimizer step
    optmizer.step()
    pass

print(loss.item())
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))
           

In [ ]:
print(sum([p.numel() for p in m.parameters()]))

In [ ]:
a = torch.tril(torch.ones(3, 3))
a = a/a.sum(dim=1, keepdim=True)
a


In [ ]:
b = torch.randint(0, 10, (3, 2)).float()
print(b)
b = b/b.sum(dim=1, keepdim=True)
b

### 0 addes up through the columns 
### 1 addes up through the rows. 
eg:  
```
import torch

a = torch.tensor([[1, 2, 3], [4, 5, 6]])

print(torch.sum(a, dim=0))  # tensor([5, 7, 9])
print(torch.sum(a, dim=1))  # tensor([6, 15])

```
- torch.allclose(...) — checks whether both tensors have approximately equal values.
- assert means "check whether this condition is true."


In [28]:
import torch
import torch.nn.functional as F
torch.manual_seed(42)

# toy example 

a = torch.tril(torch.ones(3, 3))
a = a /torch.sum(a, 1,  keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
assert torch.allclose(a.sum(1), torch.ones(3))
a, b, c



(tensor([[1.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000],
         [0.3333, 0.3333, 0.3333]]),
 tensor([[2., 7.],
         [6., 4.],
         [6., 5.]]),
 tensor([[2.0000, 7.0000],
         [4.0000, 5.5000],
         [4.6667, 5.3333]]))

In [30]:
# version 1
B, T, C = 4, 8, 2   
x = torch.randn(B, T, C)
x.shape
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)
assert torch.allclose(xbow[:, 0], x[:, 0])

# ============ VERSION 2: instead of calculating mean in for loop============

B, T, C = 4, 8, 2                                     # batch, time, channels


wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, 1, keepdim=True)
xbow2 = wei @ x
print("v2 matches loops:", torch.allclose(xbow, xbow2))


v2 matches loops: True


In [ ]:
# day3_trick.py
import torch
import torch.nn.functional as F

# ============ TOY: matmul as weighted aggregation ============
torch.manual_seed(42)
a_toy = torch.tril(torch.ones(3, 3))                  # lower-triangular ones
a_toy = a_toy / torch.sum(a_toy, 1, keepdim=True)     # keepdim=True matters: (3,1) divides each ROW
b_toy = torch.randint(0, 10, (3, 2)).float()
c_toy = a_toy @ b_toy                                 # row t = average of b_toy rows 0..t
print("a=\n", a_toy, "\nb=\n", b_toy, "\nc=\n", c_toy)
assert torch.allclose(a_toy.sum(1), torch.ones(3))    # every row of weights sums to 1

# ============ SETUP ============
torch.manual_seed(1337)
B, T, C = 4, 8, 2                                     # batch, time or token, channels or number representing each token which is 2 here
x = torch.randn(B, T, C)

# ============ VERSION 1: loops (slow, obvious) ============
# xbow is a simple way to combine the information from earlier tokens into the current token's representation.
xbow = torch.zeros((B, T, C))                         # "bag of words": average of the past
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]                          # (t+1, C): positions 0..t INCLUDING t
        xbow[b, t] = torch.mean(xprev, 0)             # average over the time dimension
assert torch.allclose(xbow[:, 0], x[:, 0])            # position 0 averages only itself

# ============ VERSION 2: one matrix multiply ============
wei = torch.tril(torch.ones(T, T))                    # (T, T) lower-triangular
wei = wei / wei.sum(1, keepdim=True)                  # rows sum to 1 -> averaging weights
xbow2 = wei @ x                                       # (T,T) @ (B,T,C) -> (B,T,C), broadcast over B or (8, 8) @ (4, 8, 2) → (4, 8, 2) or batch is not consider so (8, 8) @ (8, 2) => 8,2 for all batches its 4, 8, 2
print("v2 matches loops:", torch.allclose(xbow, xbow2))

# ============ VERSION 3: softmax with a mask ============
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))                             # "affinities": all equal for now
wei = wei.masked_fill(tril == 0, float('-inf'))       # future positions -> -inf
wei = F.softmax(wei, dim=-1)                          # exp(-inf)=0; the rest share weight equally
xbow3 = wei @ x
print("v3 matches loops:", torch.allclose(xbow, xbow3))
print(wei)                                            # same matrix as version 2's wei